# Suite de Tests - Pipeline de Vérification Basée sur les Connaissances

## Vue d'ensemble du notebook

**Objectif:** Test et validation complets du système de fact-checking basé sur les connaissances avec:
1. **Analyse de sensibilité détection de réclamations** (variations de seuil)
2. **Exemples multilingues** (EN/FR/ES)
3. **Test de robustesse récupération de preuves**
4. **Validation des verdicts de vérification**

**Environnement:** Google Colab avec intégration Google Drive

---

## Architecture de la suite de tests

```
Phase 1: Import de modèles et configuration
├─ Récupérateur de preuves (multi-source configuré)
├─ Détecteur de réclamations (modèle formé depuis knowledge.ipynb)
└─ Vérificateur de réclamations (modèle NLI DeBERTa)

Phase 2: Tests détection de réclamations
├─ Inférence réclamation unique
└─ Analyse sensibilité de seuil (0.1-0.9)

Phase 3: Tests extraction d'entités
├─ Tests NER bilingues (EN/FR/ES)
└─ Filtrage d'entités par type (PERSON, LOC, ORG, etc.)

Phase 4: Tests récupération de preuves
├─ Récupération multi-source
└─ Validation priorité de source

Phase 5: Tests vérification de réclamations
├─ Inférence d'entailment isolée
└─ Interprétation de verdict

Phase 6: Tests pipeline complet
├─ Workflow fact-checking complet
└─ Cas de test du monde réel (multilingues)

Phase 7: Agrégation et analyse des résultats
├─ Résumé performance par catégorie
├─ Analyse d'erreurs
└─ Distribution de confiance
```

---

## Paramètres de test clés

**Seuils de détection de réclamations:**
- 0.1: Très permissif (inclut opinions)
- 0.2: Conservateur (standard)
- 0.3-0.5: Plus strict (haute précision)
- 0.7+: Très strict (réclamations indéniables)

**Seuils de vérification (entailment NLI):**
- ≥ 0.70: SUPPORTÉE (fort consensus)
- ≥ 0.85: HAUTE CONFIANCE
- 0.30-0.70: NEUTRE (ambigu)
- ≤ 0.30: RÉFUTÉE (contradiction)

**Catégories de test:**
- Géographie: Assertions factuelles sur emplacements
- Science: Lois physiques, faits mathématiques
- Histoire: Faits temporels sur événements passés
- Désinformation: Faussetés connues pour valider réfutation
- Médecine: Assertions de santé (enjeux élevés)

---

## Sources de preuves testées

| Source | Type | Fiabilité | Couverture |
|--------|------|-----------|-----------|
| WolframAlpha | Mathématiques, statistiques, dates | Très haute | Domaines spécialisés |
| Google CSE | Fact-checking (PolitiFact, Le Monde) | Haute | Politique, faits vérifiés |
| Wikipédia | Connaissances générales | Moyenne-haute | Large couverture |

---

## Critères de succès

**Test réussi si:**
- Verdict attendu = verdict obtenu
- Confiance > 0.6 (signal clair)
- Preuve trouvée dans 80%+ des cas

**Performance cible:**
- Overall accuracy: ≥ 95% (production)
- Par catégorie: ≥ 80% (acceptable, amélioration possible)

---

## Temps d'exécution estimé

- **Full test suite:** 5-10 minutes (GPU)
- **Sous-ensemble (recommandé):** 2-3 minutes
- **Durée par phase:**
  - Configuration: < 1 min
  - Tests détection: 30 sec
  - Tests récupération preuves: 2-3 min
  - Tests vérification: 1-2 min
  - Pipeline complet: 1-2 min

---

## Dépannage des défaillances courantes

| Problème | Cause probable | Solution |
|---------|----------------|----------|
| Détection always False | Modèle non chargé | Vérifier chemin modèle; former si absent |
| Récupération preuves None | Clés API invalides | Configurer WOLFRAM/GOOGLE keys |
| Verdicts non-concordants | Sensibilité modèle DeBERTa | Ajuster seuils (0.70, 0.85) |
| Tests FR/ES échouent | Modèles spaCy manquants | `python -m spacy download fr_core_news_sm` |
| Google CSE: 0 résultats | Custom Search non configuré | Créer à console.cloud.google.com |
| Latence élevée (1ère run) | Téléchargements modèles | Normal; relancer sera plus rapide |

---

## Note importante

⚠️ **Duplication de code:** Ce notebook partage du code avec `knowledge.ipynb` (sections d'entraînement de modèles). Pour les détails complets, consulter `knowledge.ipynb`.

---

## Exécution recommandée

1. Exécutez **Phase 1** pour charger tous les composants
2. Exécutez **Phases 2-6** dans l'ordre pour tests progressifs
3. Exécutez **Phase 7** pour analyse complète des résultats

Chaque phase est **indépendante** une fois Phase 1 exécutée.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r drive/MyDrive/Projet_NLP.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: missing destination file operand after 'drive/MyDrive/Projet_NLP.'
Try 'cp --help' for more information.


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import os
import torch
device = 'GPU' if torch.cuda.is_available() else 'CPU'
print('Vous utilisez le {} '.format(device))

Vous utilisez le CPU 


In [ ]:
ls

drive/  sample_data/


In [ ]:
cd /content/drive/MyDrive/Projet_NLP

/content/drive/MyDrive/Projet_NLP


In [ ]:
ls

Basic_information  groundtruth.csv    Proposition_projet.pdf  unzip.py
data/              knowledge_branch/  README.md
fusion_branch/     main.ipynb         style_branch/


# Claim detection

In [ ]:
!pip install transformers torch spacy pandas
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 35.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Téléchargement du fichier "groundtruth" (le plus fiable pour le fine-tuning)
!wget https://zenodo.org/api/records/3609356/files/groundtruth.csv/content -O groundtruth.csv

--2026-03-16 23:41:14--  https://zenodo.org/api/records/3609356/files/groundtruth.csv/content
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.184.103.118, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 167741 (164K) [text/plain]
Saving to: ‘groundtruth.csv’

groundtruth.csv     100%[===================>] 163.81K   493KB/s    in 0.3s    

2026-03-16 23:41:15 (493 KB/s) - ‘groundtruth.csv’ saved [167741/167741]



In [ ]:
import pandas as pd

dataset = pd.read_csv("groundtruth.csv")
print(dataset.head())
print(dataset.columns.to_list)
print(dataset['Verdict'].value_counts())
print(dataset.shape)

# On crée une colonne 'labels' binaire : 1 pour le niveau 1, 0 pour le reste
dataset['labels'] = dataset['Verdict'].apply(lambda x: 1 if x == 1 else 0)

# On ne garde que le texte et le nouveau label
dataset = dataset[['Text', 'labels']]



   Sentence_id                                               Text  \
0           26      You know, I saw a movie - "Crocodile Dundee."   
1           80  We're consuming 50 percent of the world's coca...   
2          129   That answer was about as clear as Boston harbor.   
3          131                          Let me help the governor.   
4          172  We've run up more debt in the last eight years...   

           Speaker   Speaker_title Speaker_party         File_id  Length  \
0      George Bush  Vice President    REPUBLICAN  1988-09-25.txt       9   
1  Michael Dukakis        Governor      DEMOCRAT  1988-09-25.txt       8   
2      George Bush  Vice President    REPUBLICAN  1988-09-25.txt       9   
3      George Bush  Vice President    REPUBLICAN  1988-09-25.txt       5   
4  Michael Dukakis        Governor      DEMOCRAT  1988-09-25.txt      22   

   Line_number  Sentiment  Verdict  
0           26   0.000000        0  
1           80  -0.740979        1  
2          129   

In [ ]:
dataset.head()

,Text,labels
0,"You know, I saw a movie - ""Crocodile Dundee.""",0
1,We're consuming 50 percent of the world's coca...,1
2,That answer was about as clear as Boston harbor.,0
3,Let me help the governor.,0
4,We've run up more debt in the last eight years...,1


In [ ]:
from datasets import Dataset, DatasetDict

# On transforme le DataFrame en Dataset Hugging Face
hg_dataset = Dataset.from_pandas(dataset[['Text', 'labels']])

# on divise notre dataset en train/test

split_dataset = hg_dataset.train_test_split(test_size=0.2)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    # L'option batched=True envoie une liste de textes ici
    return tokenizer(examples["Text"], padding="max_length", truncation=True)

tokenized_datasets = split_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

# Lancer l'entraînement
trainer.train()


Map:   0%|          | 0/825 [00:00<?, ? examples/s]

Map:   0%|          | 0/207 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,No log,0.220350
2,No log,0.174156
3,No log,0.170779


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=156, training_loss=0.15872716903686523, metrics={'train_runtime': 156.6669, 'train_samples_per_second': 15.798, 'train_steps_per_second': 0.996, 'total_flos': 327856811673600.0, 'train_loss': 0.15872716903686523, 'epoch': 3.0})

In [ ]:
trainer.save_model("./my_claim_model")
tokenizer.save_pretrained("./my_claim_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_claim_model/tokenizer_config.json', './my_claim_model/tokenizer.json')

In [ ]:
from transformers import pipeline

claim_pipeline = pipeline("text-classification", model="./my_claim_model", tokenizer="./my_claim_model", device=0)


def detect_claim(text, threshold=0.2):
    results = claim_pipeline(text)
    # Dans ton modèle, vérifie si LABEL_1 est bien le "Check-worthy"
    # Si le score pour LABEL_1 est > 0.2, on y va.
    score_claim = next((r['score'] for r in results if r['label'] == 'LABEL_1'), 0)
    return score_claim > threshold, score_claim

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

# Evidence Retrieval

In [ ]:
! pip install wikipedia-api

In [ ]:
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 47.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 35.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 93.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('

In [ ]:
import wikipediaapi
import spacy
import requests

class EvidenceRetriever:
  def __init__(self, google_api_key=None, google_cse_id=None, wolfram_app_id=None,
                 config_languages={'en': 'en_core_web_sm', 'fr': 'fr_core_news_sm'}):
      self.wiki_instances = {}
      self.nlp_models = {}
      self.google_api_key = google_api_key
      self.google_cse_id = google_cse_id
      self.wolfram_app_id = wolfram_app_id

      user_agent = "FakeNewsDetectionProject/1.0 "

      for language, spacy_models in config_languages.items():
        self.wiki_instances[language] = wikipediaapi.Wikipedia(
            user_agent=user_agent, language=language)

        try:
          self.nlp_models[language] = spacy.load(spacy_models)

        except OSError:
          print(f"⚠️ Modèle {spacy_models} manquant.")

  def extract_entities(self, text, language):
      nlp = self.nlp_models.get(language)
      if not nlp:
        return text
      doc = nlp(text)
      entities = [ent.text for ent in doc.ents if ent.label_ in ['PERSON', 'LOC', 'GPE', 'ORG', 'FAC']]
      return " ".join(entities) if entities else text

  def get_wolfram_evidence(self, claim):
    if not self.wolfram_app_id:
      return None
    url = "http://api.wolframalpha.com/v1/result"
    parameters = {
        "appid": self.wolfram_app_id, "i": claim
    }
    try:
      response = requests.get(url, params=parameters)
      if response.status_code == 200:
          return {"title": "WolframAlpha", "content": response.text, "url": "https://www.wolframalpha.com"}
    except: return None

  def get_google_politifact_evidence(self, claim):
    if not self.google_api_key:
      return None
    query = f"site:politifact.com OR site:lemonde.fr {claim}"
    url = "https://www.googleapis.com/customsearch/v1"
    params = {"key": self.google_api_key, "cx": self.google_cse_id, "q": query}

    try:
      response = requests.get(url, params=params).json()
      if 'items' in response:
        return {"title": response['items'][0]['title'], "content": response['items'][0]['snippet'], "url": response['items'][0]['link']}
    except:
      return None

  def get_evidence(self, claim_text, language='en'):

    subject_query = self.extract_entities(claim_text, language)

    evidence = self.get_wolfram_evidence(subject_query)
    if evidence:
      return evidence

    evidence = self.get_google_politifact_evidence(subject_query)
    if evidence: return evidence

    wiki = self.wiki_instances.get(language)

    if wiki:
      search_url = f"https://{language}.wikipedia.org/w/api.php"
      # Ajoute un User-Agent clair ici
      headers = {"User-Agent": "FactCheckerProject/1.0 (votre_email@example.com)"}

      params = {
          "action": "opensearch",
          "search": subject_query,
          "limit": 1,
          "format": "json"
      }

      response = requests.get(search_url, params=params, headers=headers)

      # Sécurité : on vérifie que la réponse est OK (code 200)
      if response.status_code == 200:
          try:
              data = response.json()
              if len(data) > 1 and data[1]:
                  page = wiki.page(data[1][0])
                  if page.exists():
                      return {"title": page.title, "content": page.summary[:1200], "url": page.fullurl}
          except Exception as e:
              print(f"Erreur lors du décodage JSON : {e}")
      else:
          print(f"Erreur Wikipedia : Code {response.status_code}")

    return None

In [ ]:
# Remplace par tes vraies clés obtenues sur les portails
WOLFRAM_APPID = "LEU7Y6728T"
GOOGLE_API_KEY = None
GOOGLE_CSE_ID = "151bf4aa4eae44373"

# Initialisation du nouveau Retriever
retriever = EvidenceRetriever(
    google_api_key=GOOGLE_API_KEY,
    google_cse_id=GOOGLE_CSE_ID,
    wolfram_app_id=WOLFRAM_APPID,
    config_languages={'en': 'en_core_web_sm', 'fr': 'fr_core_news_sm', 'es': 'es_core_news_sm'}
)

test_claims = {
    "The Eiffel Tower is 330 meters tall": "en",
    "Le taux de chômage en France est de 7%": "fr",
    "La capital de España es Madrid": "es"
}

print("--- Lancement du Test Multi-Sources ---\n")

for claim, language in test_claims.items():
    # On utilise la méthode qui gère les priorités
    evidence = retriever.get_evidence(claim, language)

    if evidence:
        source_name = evidence.get('title', 'Inconnue')
        print(f"✅ Trouvé via [{source_name}] ({language})")
        print(f"Phrase : {claim}")
        print(f"Preuve : {evidence['content'][:150]}...")
        print(f"Lien : {evidence.get('url', 'N/A')}\n")
    else:
        print(f"❌ Aucune preuve trouvée pour : {claim}\n")

--- Lancement du Test Multi-Sources ---

✅ Trouvé via [WolframAlpha] (en)
Phrase : The Eiffel Tower is 330 meters tall
Preuve : The Eiffel Tower is a wrought–iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company d...
Lien : https://www.wolframalpha.com

✅ Trouvé via [WolframAlpha] (fr)
Phrase : Le taux de chômage en France est de 7%
Preuve : France, officially the French Republic, is a country primarily located in Western Europe....
Lien : https://www.wolframalpha.com

✅ Trouvé via [WolframAlpha] (es)
Phrase : La capital de España es Madrid
Preuve : Madrid is the capital and most populous municipality of Spain. Madrid has almost 3.3 million inhabitants and a metropolitan area population of approxi...
Lien : https://www.wolframalpha.com



# Claim Verification

In [ ]:
from transformers import pipeline

class ClaimVerifier:

  def __init__ (self):
    model_name = "MoritzLaurer/DeBERTa-v3-base-zeroshot-v1.1-all-33"
    print("Chargement du modèle de vérification")

    self.classifier = pipeline("text-classification", model=model_name, device=0)


  def verify(self, claim, evidence_text):
    if not evidence_text:
        return "NOT ENOUGH INFO", 0.0
    text_input = f"{evidence_text[:1000]} [SEP] {claim}"

    try:

      result = self.classifier(text_input)[0]

      label_detected = result['label']
      score = result['score']

      mapping = {
                "entailment": "SUPPORTED",
                "not_entailment": "REFUTED", # Ou "NEUTRAL/REFUTED"
                "LABEL_0": "SUPPORTED",
                "LABEL_1": "REFUTED"
            }
      if label_detected == "entailment" and score >= 0.60:
        #return mapping[label_detected], score
        return "SUPPORTED", score
      elif label_detected == "not_entailment" or (label_detected == "entailment" and score <= 0.85):
            # Si le score est faible, c'est probablement que la source parle du sujet
            # (Area 51) sans confirmer l'action (hiding aliens)
            return "REFUTED / UNVERIFIED", score
      else:
            return "NEUTRAL", score

    except Exception as e:
      return f"ERROR: {str(e)}", 0.0

In [ ]:
claimverifier = ClaimVerifier()
claimverifier.verify("New York is in the United States", "New York is one of the bigest city of the United States")

Chargement du modèle de vérification


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

('SUPPORTED', 0.9993588328361511)

In [ ]:
# --- CONFIGURATION ---

verifier = ClaimVerifier()
text_to_check = "Tom Cruise was born in 1962"
FORCE_CHECK = True # Pour forcer la vérification même si BERT hésite

is_claim, detection_score = detect_claim(text_to_check, threshold=0.2)

if is_claim or FORCE_CHECK:
    if not is_claim:
        print(f"⚠️ Détection faible ({detection_score:.2f}), mais on vérifie quand même...")
    else:
        print(f"🔍 Claim détecté ! (Confiance: {detection_score:.2f})")

    # ÉTAPE 2 : Récupération
    evidence = retriever.get_evidence(text_to_check, language='en')

    if evidence:
        # ÉTAPE 3 : Vérification
        verdict, confidence = verifier.verify(text_to_check, evidence['content'])
        print(f"\n--- RÉSULTAT FINAL ---")
        print(f"Phrase : {text_to_check}")
        print(f"Verdict : {verdict} (Score NLI: {confidence:.2f})")
        print(f"Source : {evidence['url']} (via {evidence.get('title')})")
    else:
        print("❌ Aucune preuve trouvée.")
else:
    print("ℹ️ Cette phrase n'a pas été jugée digne d'intérêt par le détecteur.")

Chargement du modèle de vérification


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

⚠️ Détection faible (0.00), mais on vérifie quand même...

--- RÉSULTAT FINAL ---
Phrase : Tom Cruise was born in 1962
Verdict : SUPPORTED (Score NLI: 1.00)
Source : https://www.wolframalpha.com (via WolframAlpha)


In [ ]:
retriever = EvidenceRetriever(google_api_key=None, google_cse_id=None, wolfram_app_id="LEU7Y6728T",
                 config_languages={'en': 'en_core_web_sm', 'fr': 'fr_core_news_sm'})
verifier = ClaimVerifier()

text_to_check = "New York is in the United States  "

is_claim, detection_score = detect_claim(text_to_check, threshold=0.1)

if is_claim:
  print(f"🔍 Claim détecté ! (Confiance: {detection_score:.2f})")

  evidence = retriever.get_evidence(text_to_check, language='en')
  if evidence:
    verdict, confidence = verifier.verify(text_to_check, evidence['content'])
    print(f"\n--- RÉSULTAT FINAL ---")
    print(f"Phrase : {text_to_check}")
    print(f"Verdict : {verdict} (Score NLI: {confidence:.2f})")
    print(f"Source : {evidence['url']}")
  else:
        print("❌ Aucune preuve trouvée sur Wikipedia.")
else:
    print("ℹ️ Cette phrase n'est pas considérée comme une affirmation factuelle à vérifier.")

Chargement du modèle de vérification


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

ℹ️ Cette phrase n'est pas considérée comme une affirmation factuelle à vérifier.


In [ ]:
retriever = EvidenceRetriever(google_api_key=GOOGLE_API_KEY,
    google_cse_id="151bf4aa4eae44373",
    wolfram_app_id="LEU7Y6728T",
    config_languages={'en': 'en_core_web_sm', 'fr': 'fr_core_news_sm', 'es': 'es_core_news_sm'})

verifier = ClaimVerifier()

def process_full_text(text, language='en'):
    nlp = retriever.nlp_models.get(language)
    doc = nlp(text)
    final_report = []

    for sent in doc.sents:
        sentence = sent.text.strip()
        if not sentence: continue

        # --- DÉTECTION HYBRIDE ---
        is_claim, detection_score = detect_claim(sentence, threshold=0.2)
        # On force la vérification si l'IA dit OUI ou si spaCy voit des entités
        has_entities = len([ent for ent in sent.ents if ent.label_ in ['GPE', 'PERSON', 'DATE', 'ORG']])

        if is_claim or has_entities:
            print(f"🔎 Analyse de : {sentence} (Score IA: {detection_score:.2f})")

            # --- RÉCUPÉRATION (Appel crucial) ---
            evidence = retriever.get_evidence(sentence, language)

            if evidence:
                # --- VÉRIFICATION ---
                verdict, score = verifier.verify(sentence, evidence['content'])
                final_report.append({
                    "phrase": sentence,
                    "verdict": verdict,
                    "confiance": score,
                    "source": evidence['url'],
                    "source_name": evidence.get('title')
                })
            else:
                final_report.append({"phrase": sentence, "verdict": "NO EVIDENCE FOUND"})
        else:
            print(f"⏭️ Ignoré : {sentence}")

    return final_report

Chargement du modèle de vérification


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [ ]:
mon_article = """
Tom Cruise is an american actor. The Eiffel Tower is in Paris. The Eiffel tower was built in 1990.

"""

resultats = process_full_text(mon_article, language='en')

# Affichage propre
for r in resultats:
    print(f"[{r['verdict']}] {r['phrase']} (Source: {r.get('source', 'N/A')})")

🔎 Analyse de : Tom Cruise is an american actor. (Score IA: 0.00)
🔎 Analyse de : The Eiffel Tower is in Paris. (Score IA: 0.56)
🔎 Analyse de : The Eiffel tower was built in 1990. (Score IA: 0.69)
[SUPPORTED] Tom Cruise is an american actor. (Source: https://www.wolframalpha.com)
[SUPPORTED] The Eiffel Tower is in Paris. (Source: https://www.wolframalpha.com)
[REFUTED / UNVERIFIED] The Eiffel tower was built in 1990. (Source: https://www.wolframalpha.com)
